In [1]:
"""
08_statistical_validation.py
------------------------------
Phase 10: formal chi-square tests of independence, checking
whether CLV tier (High/Medium/Low) is significantly associated
with acquisition_channel, region, and age_band.

Channel is expected to come out highly significant (matches the
behavioral pattern designed into the data and seen in feature
importance). Region/age_band are expected to NOT be significant,
since the model gave them low importance - this contrast is
itself a useful, honest part of the story.

Reads:
    data/processed/segmented_customers.csv

Writes:
    data/processed/chi_square_results.csv
"""

import pandas as pd
from scipy.stats import chi2_contingency

df = pd.read_csv("data/processed/segmented_customers.csv")
print(f"Loaded {len(df)} customers\n")

results = []

def run_chi_square(variable_name):
    contingency_table = pd.crosstab(df["clv_tier"], df[variable_name])
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)

    print(f"--- Chi-square test: clv_tier vs {variable_name} ---")
    print(f"Chi2 statistic: {chi2:.2f}")
    print(f"Degrees of freedom: {dof}")
    print(f"p-value: {p_value:.6f}")

    significant = p_value < 0.05
    print(f"Significant at alpha=0.05? {'YES' if significant else 'no'}\n")

    results.append({
        "variable": variable_name,
        "chi2_statistic": round(chi2, 2),
        "degrees_of_freedom": dof,
        "p_value": p_value,
        "significant_at_0.05": significant,
    })

# ----------------------------------------------------------------
# Run the test for each demographic/channel variable
# ----------------------------------------------------------------
run_chi_square("acquisition_channel")
run_chi_square("region")
run_chi_square("age_band")

# ----------------------------------------------------------------
# Save results
# ----------------------------------------------------------------
results_df = pd.DataFrame(results)
print("Summary of all tests:")
print(results_df.to_string(index=False))

results_df.to_csv("data/processed/chi_square_results.csv", index=False)
print("\nSaved: data/processed/chi_square_results.csv")

Loaded 4000 customers

--- Chi-square test: clv_tier vs acquisition_channel ---
Chi2 statistic: 3591.54
Degrees of freedom: 8
p-value: 0.000000
Significant at alpha=0.05? YES

--- Chi-square test: clv_tier vs region ---
Chi2 statistic: 14.20
Degrees of freedom: 6
p-value: 0.027449
Significant at alpha=0.05? YES

--- Chi-square test: clv_tier vs age_band ---
Chi2 statistic: 6.90
Degrees of freedom: 8
p-value: 0.547922
Significant at alpha=0.05? no

Summary of all tests:
           variable  chi2_statistic  degrees_of_freedom  p_value  significant_at_0.05
acquisition_channel         3591.54                   8 0.000000                 True
             region           14.20                   6 0.027449                 True
           age_band            6.90                   8 0.547922                False

Saved: data/processed/chi_square_results.csv
